In [ ]:
# Example 1: the management client, a VM update body and a VM instance view
# Nothing here contacts Azure: building the client and the models is local.
from azure.core.credentials import AccessToken
from azure.mgmt.compute import ComputeManagementClient
from azure.mgmt.compute.models import (
    HardwareProfile, InstanceViewStatus, VirtualMachineInstanceView, VirtualMachineUpdate,
)


def rest_body(model):
    """The JSON body the SDK sends for a model."""
    # Newer SDK models are mappings whose as_dict() is the REST body; msrest-based models use serialize().
    serialize = getattr(model, "serialize", None)
    return serialize() if serialize is not None else model.as_dict()


class FakeCredential:
    def get_token(self, *scopes, **kwargs):
        raise AssertionError("no request should need a token in this notebook")


client = ComputeManagementClient(FakeCredential(), subscription_id="00000000-0000-0000-0000-000000000000")
for operation in ("begin_create_or_update", "begin_update", "begin_start", "begin_power_off",
                  "begin_deallocate", "instance_view", "list"):
    assert callable(getattr(client.virtual_machines, operation, None)), operation
client.close()

update = VirtualMachineUpdate(tags={"env": "test"}, hardware_profile=HardwareProfile(vm_size="Standard_DS2_v2"))
body = rest_body(update)
assert body == {"tags": {"env": "test"}, "properties": {"hardwareProfile": {"vmSize": "Standard_DS2_v2"}}}, body
print("VM update body:", body)

view = VirtualMachineInstanceView(statuses=[
    InstanceViewStatus(code="ProvisioningState/succeeded", level="Info"),
    InstanceViewStatus(code="PowerState/running", level="Info", display_status="VM running"),
])
power = [s.code.split("/", 1)[1] for s in view.statuses if s.code.startswith("PowerState/")]
assert power == ["running"], power
assert rest_body(view)["statuses"][1] == {"code": "PowerState/running", "level": "Info", "displayStatus": "VM running"}
print("power state from the instance view:", power[0])

In [ ]:
# Example 2: an empty managed disk and the managed-disk parameters of a VM disk
from azure.mgmt.compute.models import (
    CreationData, Disk, DiskCreateOption, DiskSku, ManagedDiskParameters, StorageAccountTypes,
)

disk = Disk(
    location="eastus",
    creation_data=CreationData(create_option=DiskCreateOption.EMPTY),
    disk_size_gb=128,
    sku=DiskSku(name=StorageAccountTypes.PREMIUM_LRS),
)
assert disk.disk_size_gb == 128 and disk.sku.name == "Premium_LRS"
assert disk.creation_data.create_option == DiskCreateOption.EMPTY == "Empty"
body = rest_body(disk)
assert body["location"] == "eastus" and body["sku"] == {"name": "Premium_LRS"}, body
assert body["properties"]["diskSizeGB"] == 128, body
assert body["properties"]["creationData"] == {"createOption": "Empty"}, body
print("disk body:", body)

managed = ManagedDiskParameters(storage_account_type=StorageAccountTypes.STANDARD_SSD_LRS)
assert rest_body(managed) == {"storageAccountType": "StandardSSD_LRS"}, rest_body(managed)
print("managed disk parameters:", rest_body(managed))

In [ ]:
# Example 3: a scale set with an automatic upgrade policy, then scaled from 3 to 5 instances
from azure.mgmt.compute.models import Sku, UpgradeMode, UpgradePolicy, VirtualMachineScaleSet

assert {"Automatic", "Manual", "Rolling"} <= {mode.value for mode in UpgradeMode}
scale_set = VirtualMachineScaleSet(
    location="eastus",
    sku=Sku(name="Standard_DS1_v2", tier="Standard", capacity=3),
    upgrade_policy=UpgradePolicy(mode=UpgradeMode.AUTOMATIC),
)
assert scale_set.upgrade_policy.mode == "Automatic" and scale_set.sku.capacity == 3
scale_set.sku.capacity = 5  # a scale-out is an update of the sku capacity
body = rest_body(scale_set)
assert body["sku"] == {"name": "Standard_DS1_v2", "tier": "Standard", "capacity": 5}, body
assert body["properties"]["upgradePolicy"] == {"mode": "Automatic"}, body
print("scale set body:", body)